# Customer Churn Prediction System
**MSc Data Science Capstone — University of Pittsburgh**

**Business Goal:** Identify customers likely to churn so the business can intervene before losing them.

**Dataset:** Telco Customer Churn (IBM Sample) — [Kaggle link](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
> Download `WA_Fn-UseC_-Telco-Customer-Churn.csv` and place it in `../data/`

---
**Workflow:** Data Cleaning → EDA → Feature Engineering → Modeling → Evaluation → Model Export

## Section 1 — Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, RocCurveDisplay
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

print("Libraries loaded successfully")

## Section 2 — Load and Inspect Dataset

In [ ]:
DATA_PATH = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(DATA_PATH)

print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
df.head()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nChurn distribution:\n{df['Churn'].value_counts()}")

## Section 3 — Data Cleaning and Preprocessing

In [ ]:
# Drop customerID — not a predictor
df.drop(columns=['customerID'], inplace=True)

# TotalCharges is stored as string — convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Drop rows where TotalCharges could not be parsed (11 rows with empty string)
df.dropna(subset=['TotalCharges'], inplace=True)
df.drop_duplicates(inplace=True)

# Encode binary target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Encode binary Yes/No columns
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
               'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'No phone service': 0,
                           'No internet service': 0})

# Encode SeniorCitizen is already 0/1
# One-hot encode remaining categoricals
df = pd.get_dummies(df, columns=['gender', 'InternetService', 'Contract', 'PaymentMethod'],
                    drop_first=False)

print(f"Clean dataset shape: {df.shape}")
df.head(3)

## Section 4 — Exploratory Data Analysis (EDA)

In [ ]:
# 4a. Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df['Churn'].value_counts()
axes[0].bar(['No Churn', 'Churn'], churn_counts, color=['#2ecc71', '#e74c3c'])
axes[0].set_title('Churn Distribution (Count)')
axes[0].set_ylabel('Number of Customers')

axes[1].pie(churn_counts, labels=['No Churn', 'Churn'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[1].set_title('Churn Distribution (%)')

plt.suptitle('Customer Churn Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Churn rate: {df['Churn'].mean()*100:.1f}%")

In [ ]:
# 4b. Distribution of key numeric features by churn
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    df[df['Churn'] == 0][col].hist(bins=30, ax=ax, alpha=0.6, color='#2ecc71', label='No Churn')
    df[df['Churn'] == 1][col].hist(bins=30, ax=ax, alpha=0.6, color='#e74c3c', label='Churn')
    ax.set_title(col)
    ax.legend()
plt.suptitle('Numeric Feature Distributions by Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4c. Correlation heatmap (top numeric columns)
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='RdYlGn', center=0,
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 5 — Feature Engineering

In [ ]:
# New engineered features
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)   # avoid div/0
df['HasMultipleServices'] = (
    df[['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies']].sum(axis=1) >= 3
).astype(int)

# Separate features and target
TARGET = 'Churn'
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Train / test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numeric features
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend']
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

print(f"Train size: {X_train.shape}  |  Test size: {X_test.shape}")
print(f"Features used: {X_train.shape[1]}")

## Section 6 — Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting':    GradientBoostingClassifier(n_estimators=100, random_state=42),
}

trained = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    print(f"  Trained: {name}")

print("\nAll models trained.")

## Section 7 — Model Evaluation

In [ ]:
# Comparison table
results = []
for name, model in trained.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred), 4),
        'F1':        round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob), 4),
    })

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, model) in zip(axes, trained.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves — all models on one chart
fig, ax = plt.subplots(figsize=(8, 6))
for name, model in trained.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=name)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax.set_title('ROC Curves — Model Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../images/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance (Random Forest)
rf_model = trained['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top20 = importances.nlargest(20)

plt.figure(figsize=(10, 6))
top20.sort_values().plot(kind='barh', color='#4299e1')
plt.title('Top 20 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../images/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 8 — Save Best Model as .pkl

In [ ]:
# Select best model by ROC-AUC
best_name = results_df.iloc[0]['Model']
best_model = trained[best_name]
print(f"Best model: {best_name}  (ROC-AUC = {results_df.iloc[0]['ROC-AUC']})")

# Save model and scaler
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/churn_model.pkl')
joblib.dump(scaler,     '../models/scaler.pkl')
joblib.dump(list(X_train.columns), '../models/feature_columns.pkl')

print("Saved:")
print("  ../models/churn_model.pkl")
print("  ../models/scaler.pkl")
print("  ../models/feature_columns.pkl")